# MNIST Data Preparation
Load MNIST dataset and save to volume for model training

In [ ]:
%pip install --quiet mlflow==3.6.0 tensorflow==2.20.0 scikit-learn==1.8.0
dbutils.library.restartPython()

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
import logging
from sklearn.model_selection import train_test_split

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Get volume_path from DAB parameter (or use default)
try:
    volume_path = dbutils.widgets.get('volume_path')
except:
    volume_path = '/Volumes/ai_ml_in_practice/mnist_data/processed'

print(f'Volume path: {volume_path}\n')

seed = int(dbutils.widgets.get('random_state'))
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f'Raw shapes - train: {x_train.shape}, test: {x_test.shape}')
print(f'Train labels unique: {np.unique(y_train)}')
print(f'Class distribution: {np.bincount(y_train)}')

In [ ]:
# Flatten images to 784-d vectors for sklearn models
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

print(f'Flattened shapes - train: {x_train_flat.shape}, test: {x_test_flat.shape}')

# Create train/val splits (80/20 of training data)
x_train_split, x_val, y_train_split, y_val = train_test_split(
    x_train_flat, y_train,
    test_size=0.2,
    random_state=seed,
    stratify=y_train
)

print(f'After split - train: {x_train_split.shape}, val: {x_val.shape}, test: {x_test_flat.shape}')


In [ ]:
# Volume already created by DAB (uc_resources.yml)
# Just save MNIST data to the configured volume

# Save train/val/test splits
splits = {
    'x_train': x_train_split,
    'y_train': y_train_split,
    'x_val': x_val,
    'y_val': y_val,
    'x_test': x_test_flat,
    'y_test': y_test,
    'x_train_images': x_train,
    'y_train_images': y_train,
    'x_test_images': x_test,
    'y_test_images': y_test
}

for key, arr in splits.items():
    path = os.path.join(volume_path, f'{key}.npy')
    np.save(path, arr)
    logger.info(f'Saved {key}: {arr.shape}')

print(f'All MNIST data saved to {volume_path}')